### [Pandas to Detect Anomalies in Time-Series Data Without ML](https://medium.com/@bhagyarana80/how-i-used-pandas-to-detect-anomalies-in-time-series-data-without-ml-7c1dbe8c736e)

> Windowed averages, IQR, z-score, and rolling variance techniques

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# Simulated time-series data
np.random.seed(42)
timestamps = pd.date_range(start='2023-01-01', periods=1000, freq='H')
signal = np.sin(np.linspace(0, 20, 1000)) + np.random.normal(0, 0.2, 1000)

# Inject anomalies
signal[300] += 5
signal[600] -= 4
signal[850] += 3

df = pd.DataFrame({'timestamp': timestamps, 'value': signal})
df.set_index('timestamp', inplace=True)

display(df.sample(10))

,value
timestamp,
2023-01-23 13:00:00,-1.212402
2023-01-20 10:00:00,0.218137
2023-01-08 13:00:00,-0.635011
2023-01-14 20:00:00,0.370672
2023-01-02 16:00:00,0.865607
2023-02-07 02:00:00,-0.694096
2023-01-04 08:00:00,0.955591
2023-01-09 06:00:00,-0.721120
2023-01-22 01:00:00,-0.730355


#### Windowed (Rolling) Averages

In [2]:
df['rolling_mean'] = df['value'].rolling(window=24).mean()
df['rolling_std'] = df['value'].rolling(window=24).std()

# Flag values > 3 std deviations from rolling mean
df['anomaly_rolling'] = (abs(df['value'] - df['rolling_mean']) > 3 * df['rolling_std'])

display(df.sample(10))

,value,rolling_mean,rolling_std,anomaly_rolling
timestamp,,,,
2023-02-06 22:00:00,-0.779720,-0.887110,0.219281,False
2023-01-18 13:00:00,1.190561,0.967521,0.182785,False
2023-01-20 18:00:00,0.264327,0.135041,0.239357,False
2023-01-14 04:00:00,0.179518,0.097343,0.969616,False
2023-01-04 07:00:00,0.602428,0.962760,0.213644,False
2023-02-06 08:00:00,-1.259940,-0.852643,0.619207,False
2023-01-25 17:00:00,-0.573268,-0.762304,0.203848,False
2023-02-04 20:00:00,-0.546567,-0.676540,0.243656,False
2023-01-16 06:00:00,0.909312,0.711480,0.209596,False


#### IQR-Based Outlier Detection

In [3]:
Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['anomaly_iqr'] = (df['value'] < lower_bound) | (df['value'] > upper_bound)

display(df.sample(10))

,value,rolling_mean,rolling_std,anomaly_rolling,anomaly_iqr
timestamp,,,,,
2023-01-11 09:00:00,-0.882079,-0.998440,0.190256,False,False
2023-01-15 14:00:00,0.724435,0.431819,0.138311,False,False
2023-02-10 12:00:00,0.740917,0.430132,0.204768,False,False
2023-01-01 13:00:00,-0.125324,NaN,NaN,False,False
2023-01-02 09:00:00,0.402096,0.330386,0.246087,False,False
2023-02-09 00:00:00,-0.314837,-0.336529,0.207616,False,False
2023-01-31 02:00:00,0.680448,0.949128,0.209680,False,False
2023-01-12 21:00:00,-0.936332,-0.754332,0.296495,False,False
2023-01-07 04:00:00,0.282070,0.363630,0.247360,False,False


#### Z-Score Standardization

In [4]:
mean_val = df['value'].mean()
std_val = df['value'].std()

df['z_score'] = (df['value'] - mean_val) / std_val
df['anomaly_zscore'] = abs(df['z_score']) > 3

display(df.sample(10))

,value,rolling_mean,rolling_std,anomaly_rolling,anomaly_iqr,z_score,anomaly_zscore
timestamp,,,,,,,
2023-01-05 20:00:00,0.723705,0.844727,0.175210,False,False,0.917986,False
2023-01-07 08:00:00,-0.037615,0.285342,0.209023,False,False,-0.101063,False
2023-01-04 08:00:00,0.955591,0.972045,0.207980,False,False,1.228372,False
2023-01-05 00:00:00,0.998210,0.959446,0.200664,False,False,1.285418,False
2023-01-17 23:00:00,0.928568,0.991592,0.207671,False,False,1.192201,False
2023-01-14 20:00:00,0.370672,0.184271,0.194176,False,False,0.445441,False
2023-01-01 22:00:00,0.439844,NaN,NaN,False,False,0.538029,False
2023-01-01 14:00:00,-0.068359,NaN,NaN,False,False,-0.142214,False
2023-01-23 14:00:00,-0.883585,-0.994474,0.180589,False,False,-1.233418,False


#### Rolling Variance Spikes

In [5]:
df['rolling_var'] = df['value'].rolling(window=12).var()
threshold = df['rolling_var'].mean() + 3 * df['rolling_var'].std()

df['anomaly_var'] = df['rolling_var'] > threshold

display(df.sample(10))

,value,rolling_mean,rolling_std,anomaly_rolling,anomaly_iqr,z_score,anomaly_zscore,rolling_var,anomaly_var
timestamp,,,,,,,,,
2023-01-06 19:00:00,0.104987,0.537141,0.255878,False,False,0.089814,False,0.035118,False
2023-02-09 23:00:00,0.432104,0.129839,0.251367,False,False,0.527670,False,0.029959,False
2023-02-01 15:00:00,0.630653,0.648442,0.241776,False,False,0.793433,False,0.048735,False
2023-01-30 03:00:00,0.952788,0.946079,0.176414,False,False,1.224619,False,0.022727,False
2023-01-24 02:00:00,-0.927921,-1.015767,0.179078,False,False,-1.292763,False,0.040995,False
2023-01-01 10:00:00,0.106182,NaN,NaN,False,False,0.091413,False,NaN,False
2023-01-01 04:00:00,0.033164,NaN,NaN,False,False,-0.006324,False,NaN,False
2023-01-30 18:00:00,0.980154,0.973401,0.206087,False,False,1.261250,False,0.060769,False
2023-01-18 06:00:00,0.973625,0.966507,0.207499,False,False,1.252510,False,0.028298,False


In [6]:
display(df.describe())

,value,rolling_mean,rolling_std,z_score,rolling_var
count,1000.000000,977.000000,977.000000,1.000000e+03,989.000000
mean,0.037888,0.027073,0.261491,1.421085e-17,0.087253
std,0.747089,0.686608,0.163927,1.000000e+00,0.255723
min,-4.375000,-1.034328,0.123953,-5.906777e+00,0.003986
25%,-0.630999,-0.690206,0.195972,-8.953247e-01,0.028267
50%,0.112457,0.090175,0.218047,9.981200e-02,0.040892
75%,0.676528,0.662415,0.250336,8.548374e-01,0.053821
max,4.560557,1.020645,1.051004,6.053722e+00,2.048534


In [7]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1000 entries, 2023-01-01 00:00:00 to 2023-02-11 15:00:00
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   value            1000 non-null   float64
 1   rolling_mean     977 non-null    float64
 2   rolling_std      977 non-null    float64
 3   anomaly_rolling  1000 non-null   bool   
 4   anomaly_iqr      1000 non-null   bool   
 5   z_score          1000 non-null   float64
 6   anomaly_zscore   1000 non-null   bool   
 7   rolling_var      989 non-null    float64
 8   anomaly_var      1000 non-null   bool   
dtypes: bool(4), float64(5)
memory usage: 50.8 KB


None